# Gundam NVIDIA Newton Physics Simulation (Humanoid Real Dynamics)

This notebook modifies the Gundam URDF to function as a properly scaled (1.8m) humanoid with a dynamic floating base. It sets up proper physical collisions and controls the joints via PD tracking (forward dynamics physics instead of pure kinematics teleportation). It renders the output directly to an `.mp4` video saved to your Google Drive using Newton's headless EGL/GL Viewer.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install EGL dependencies for headless rendering in Colab
!apt-get update -y
!apt-get install -y libgl1-mesa-glx xvfb libxrender1
!pip install "newton[examples]" pandas imageio imageio-ffmpeg

In [ ]:
import os
# Start virtual framebuffer to allow headless GL rendering
os.system('/usr/bin/Xvfb :99 -screen 0 1024x768x24 &')
os.environ['DISPLAY'] = ':99'

import newton
import warp as wp
import numpy as np
import pandas as pd
import imageio
from newton.solvers import SolverMuJoCo
from newton.viewer import ViewerGL

wp.init()

In [ ]:
# The URDF from the repo needs to be processed to scale it down to 1.8m human scale
# and remove the fixed anchor so it is a proper dynamic floating-base humanoid.
import xml.etree.ElementTree as ET

tree = ET.parse('/content/gundam_robot/gundam_rx78_description/urdf/GGC_TestModel_rx78_20170112.urdf')
root = tree.getroot()

# 1. Un-anchor the torso from the world. In original URDF, both torso_waist_y and rx78_Null_083_joint attached to base_link.
for j in root.findall('joint'):
    if j.get('name') == 'torso_waist_y':
        j.find('parent').set('link', 'rx78_Null_083_link')
        j.find('origin').set('xyz', '1e-07 0.0 0.5')
        j.find('origin').set('rpy', '0.0 -0.0 0.0')
        
for j in root.findall('joint'):
    if j.get('name') == 'rx78_Null_083_joint':
        root.remove(j)

for l in root.findall('link'):
    if l.get('name') == 'base_link':
        root.remove(l)

# 2. Scale down to 1.8m (1/10th scale)
SCALE = 0.1
def scale_xyz(xyz_str):
    if xyz_str is None: return None
    return ' '.join([str(float(x) * SCALE) for x in xyz_str.split()])

for j in root.findall('joint'):
    origin = j.find('origin')
    if origin is not None:
        origin.set('xyz', scale_xyz(origin.get('xyz')))

for l in root.findall('link'):
    inertial = l.find('inertial')
    if inertial is not None:
        origin = inertial.find('origin')
        if origin is not None:
            origin.set('xyz', scale_xyz(origin.get('xyz')))
        mass = inertial.find('mass')
        if mass is not None:
            m = float(mass.get('value')) * (SCALE**3)
            mass.set('value', str(m))
        inertia = inertial.find('inertia')
        if inertia is not None:
            for a in ['ixx','iyy','izz','ixy','ixz','iyz']:
                val = float(inertia.get(a, '0.0')) * (SCALE**5)
                inertia.set(a, str(val))
    
    for tag in ['visual', 'collision']:
        for elem in l.findall(tag):
            origin = elem.find('origin')
            if origin is not None:
                origin.set('xyz', scale_xyz(origin.get('xyz')))
            geom = elem.find('geometry')
            if geom is not None:
                mesh = geom.find('mesh')
                if mesh is not None:
                    mesh.set('scale', scale_xyz(mesh.get('scale', '1 1 1')))
                    # Update mesh package paths for colab
                    mesh.set('filename', mesh.get('filename').replace('package://gundam_rx78_description', '/content/gundam_robot/gundam_rx78_description'))

tree.write('/content/gundam_robot/gundam_rx78_description/urdf/humanoid_gundam.urdf')
print("Processed humanoid_gundam.urdf successfully.")

In [ ]:
# Load the sample CSV motion
csv_path = "/content/gundam_robot/gundam_rx78_control/sample/csv/walk-forward.csv"
df = pd.read_csv(csv_path)
df.columns = df.columns.str.strip()

# Extract time and normalize to 0
time_col = df['time'].values
end_time = time_col[-1] - time_col[0]
time_col = time_col - time_col[0]

# Build Newton Model from URDF
print("Parsing URDF and building model...")
builder = newton.ModelBuilder()
builder.add_urdf(
    source="/content/gundam_robot/gundam_rx78_description/urdf/humanoid_gundam.urdf", 
    ignore_inertial_definitions=True, # Auto-generate inertial properties from collision meshes
    floating=True,                    # True floating base physics
    force_position_velocity_actuation=True # Treat joints as PD controlled
)
model = builder.finalize()

state = model.state()
control = model.control()
contacts = model.contacts()

# Initialize dynamic base position
init_q = state.joint_q.numpy()
init_q[0:3] = [0.0, 0.0, 1.05] # Lift pelvis to ground level
init_q[3:7] = [0.0, 0.0, 0.0, 1.0] # Identity rotation
state.joint_q = wp.array(init_q, dtype=wp.float32)
newton.eval_fk(model, state.joint_q, state.joint_qd, state) # Apply FK initialization

# Setup PD Controller Gains for physics
k_p = 500.0
k_d = 20.0
target_ke = np.ones(model.joint_coord_count, dtype=np.float32) * k_p
target_kd = np.ones(model.joint_dof_count, dtype=np.float32) * k_d
model.joint_target_ke = wp.array(target_ke, dtype=wp.float32)
model.joint_target_kd = wp.array(target_kd, dtype=wp.float32)

# Map CSV columns to Newton joint DOFs
joint_names = builder.joint_label
csv_to_newton_idx = {}
for csv_joint in df.columns:
    if csv_joint != 'time':
        try:
            idx = joint_names.index(csv_joint)
            dof_start = builder.joint_q_start[idx]
            csv_to_newton_idx[csv_joint] = dof_start
        except ValueError:
            pass

# Pre-extract joint target arrays for fast interpolation (convert degrees to radians)
joints_list = list(csv_to_newton_idx.keys())
dof_indices = [csv_to_newton_idx[j] for j in joints_list]
joint_data = np.radians(df[joints_list].values)

# Initialize Solver (we use mujoco_contacts=False to rely on robust Newton collision)
solver = SolverMuJoCo(model, use_mujoco_contacts=False)

# Setup GL Viewer in Headless mode
fps = 60
viewer = ViewerGL(headless=True, width=640, height=480)
viewer.set_model(model)
viewer.set_camera(pos=(2.5, -2.0, 1.5), pitch=10.0, yaw=140.0) # Adjusted camera for 1.8m robot

print(f"Starting physics simulation and rendering {end_time:.2f} seconds of motion...")
sim_dt = 1.0 / 200.0
render_dt = 1.0 / float(fps)
render_accum = 0.0
current_time = 0.0

frames = []

while current_time <= end_time:
    # Interpolate target positions from CSV over continuous time
    idx1 = np.searchsorted(time_col, current_time)
    if idx1 == 0:
        interp_positions = joint_data[0]
    elif idx1 >= len(time_col):
        interp_positions = joint_data[-1]
    else:
        idx0 = idx1 - 1
        t0, t1 = time_col[idx0], time_col[idx1]
        alpha = (current_time - t0) / (t1 - t0) if t1 > t0 else 0.0
        interp_positions = (1.0 - alpha) * joint_data[idx0] + alpha * joint_data[idx1]
    
    target_positions = control.joint_target_q.numpy()
    for i, dof_idx in enumerate(dof_indices):
        target_positions[dof_idx] = interp_positions[i]
    
    # Set PD Controller targets instead of overriding physical state directly
    control.joint_target_q = wp.array(target_positions, dtype=wp.float32)
    
    # Step actual physics dynamics
    solver.step(state_in=state, state_out=state, control=control, contacts=contacts, dt=sim_dt)
    
    render_accum += sim_dt
    if render_accum >= render_dt:
        viewer.begin_frame(time=current_time)
        viewer.log_state(state=state)
        viewer.end_frame()
        
        # Capture RGB array from OpenGL context
        img_wp = viewer.get_frame()
        frames.append(img_wp.numpy())
        render_accum -= render_dt

    current_time += sim_dt
    
viewer.close()

# Save Video to Google Drive
video_path = "/content/drive/MyDrive/gundam_humanoid_physics.mp4"
if len(frames) > 0:
    imageio.mimwrite(video_path, frames, fps=fps, macro_block_size=None)
    print(f"Video saved to {video_path}")
else:
    print("No frames were captured.")
